# RENER

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import json
from sklearn.model_selection import train_test_split

# Load jsonl file into a list of dictionaries
with open("../data/labelled/all_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

# Create test split (notice: fixed seed)
train_data, test_data = train_test_split(data, test_size=0.3, random_state=51)

In [ ]:
# TODO: remove temp

data = data[:100]

In [ ]:
# Conll2003
# import json

# with open("../data/conll2003/train_parallel.json") as train:
#     train_data = json.load(train)

# with open("../data/conll2003/test_parallel.json") as test:
#     test_data = json.load(test)

In [ ]:
from collections import Counter
from util import extract_ner_labels, flatten
import matplotlib.pyplot as plt

entities = []
for item in data:
    entities.append(extract_ner_labels(item["text"]))

entities = flatten(entities)
entities_dist = Counter([ent[1] for ent in entities])

plt.figure()
plt.pie(
    entities_dist.values(),  # type: ignore
    labels=entities_dist.keys(),  # type: ignore
    autopct=lambda pct: f"{pct:.0f}%" if pct > 1.1 else "",
    startangle=90,
    pctdistance=0.5,
    wedgeprops={"edgecolor": "black"},
)

plt.title("Distribution of NER labels")
plt.show()

In [ ]:
from pathlib import Path
from time import time
from util import eval_ner, extract_ner_labels
import rich
from tqdm.notebook import tqdm

from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack.components.rankers import SentenceTransformersSimilarityRanker

from prompts import guidelines_llama3_legal_lite, default_examples
from datetime import datetime
from sklearn.model_selection import KFold


# Set parameters
model = "llama3.1:8b"
temperature = 0

# Load model using Ollama
llm = OllamaGenerator(
    model=model,
    url="http://localhost:11434",
    generation_kwargs={"temperature": temperature},
)

# Create results file for the experiment
dt = str(datetime.now()).split(".")[0].replace(" ", "_").replace(":", "-")
output_filename = Path(f"../output/rener_{dt}.json")
results = {
    "model": model,
    "temperature": temperature,
}


n_splits = 10
kf = KFold(n_splits=n_splits)
for top_k in range(2, 10):
    print(f"Running experiment with top_k={top_k}")

    # Init results dict for current top_k value
    results[f"top_k={top_k}"] = {}

    # Perform kfold cross-eval for current split and top_k
    for i, (train_index, test_index) in enumerate(kf.split(data)):
        print(f"Running split # {i + 1}/{n_splits}")

        # Initialize results dictionaries
        results[f"top_k={top_k}"][f"kfold={i + 1}"] = {}  # type: ignore
        results[f"top_k={top_k}"][f"kfold={i + 1}"]["retr_scores"] = []  # type: ignore
        results[f"top_k={top_k}"][f"kfold={i + 1}"]["eval_scores"] = []  # type: ignore

        # Split into train and test data
        train_data = [data[i] for i in train_index]
        test_data = [data[i] for i in test_index]

        # Track runtime
        start_time = time()

        y_true, y_pred, scores = [], [], []
        pbar = tqdm(test_data)
        for item in pbar:
            # Define query
            query = f"Input: {item['input']}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"

            # Setup document store with new "train" data
            document_store = InMemoryDocumentStore()
            docs = [
                Document(content=d["input"], meta={"output": d["text"]})
                for d in train_data
            ]
            document_store.write_documents(docs)

            # Run retrieval
            retriever = InMemoryBM25Retriever(document_store=document_store)
            retrieval_result = retriever.run(query=query, top_k=top_k)
            retrieved_docs = retrieval_result["documents"]

            # Format retrieved docs using llama's prompt format
            context = "".join(
                [
                    f"Input: {doc.content}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nOutput: {doc.meta['output']}<|eot_id|>\n\n"
                    for _, doc in enumerate(retrieved_docs)
                ]
            ).strip()

            # Build prompt
            prompt = guidelines_llama3_legal_lite + default_examples + context + query

            # Run the LLM and print output
            generation_result = llm.run(
                prompt=prompt,
                generation_kwargs={"temperature": temperature},
            )

            # Save prediction to pred column in dataset
            llm_output = generation_result["replies"][0]
            item["pred"] = llm_output.replace("Output: \n", "")

            # Extract natural text into labeled annotations
            true = extract_ner_labels(item["text"])
            pred = extract_ner_labels(item["pred"])

            # Apply preprocessing to account for LLMs' rewriting
            true = [(w[0].strip().title(), w[1]) for w in true]
            pred = [(w[0].strip().title(), w[1]) for w in pred]

            # Evaluate the pipeline
            y_true.append(true)
            y_pred.append(pred)
            curr_score = eval_ner(pred, true)
            scores.append(curr_score)

            # Generate metrics to show in pbar and to save to output
            avg_metrics_fold = {
                key: sum(d[key] for d in scores) / len(scores) for key in scores[0]
            }
            pbar.set_postfix(avg_metrics_fold)

            results[f"top_k={top_k}"][f"kfold={i + 1}"]["retr_scores"].append(  # type: ignore
                [d.score for d in retrieved_docs]
            )

            results[f"top_k={top_k}"][f"kfold={i + 1}"]["eval_scores"].append(scores)  # type: ignore

            # -- Debug Information --
            # print("--- Context ---")
            # print(context)
            # print()
            print("--- LLM I/O ---")
            print(query)
            print(llm_output)
            print()
            print("--- Evaluation ---")
            print(curr_score)
            print()
            print("---")
            print()

        # Store runtime and average metrics for current split
        runtime = time() - start_time
        results[f"top_k={top_k}"][f"kfold={i + 1}"]["avg_metrics_fold"] = (  # type: ignore
            avg_metrics_fold  # type: ignore
        )
        results[f"top_k={top_k}"][f"kfold={i + 1}"]["runtime"] = runtime  # type: ignore

    # Compute average metrics for each topk
    tmp = [
        results[f"top_k={top_k}"][f"kfold={i + 1}"]["avg_metrics_fold"]  # type: ignore
        for i in range(n_splits)
    ]
    avg_metrics_topk = {key: sum(d[key] for d in tmp) / len(tmp) for key in tmp[0]}
    results[f"top_k={top_k}"]["avg_metrics_topk={top_k}"] = avg_metrics_topk  # type: ignore

with open(output_filename, "w") as file:
    json.dump(results, file)


In [ ]:
results

In [ ]:
from itertools import chain
from util import extract_ner_labels
from util import eval_ner

# Flatten lists to evaluate metrics on whole dataset
flat_preds = list(map(tuple, chain.from_iterable(y_pred)))
flat_golds = list(map(tuple, chain.from_iterable(y_true)))

# Evaluate quality of extracted entities
print("--- Entity-level evaluation results ---")
scores = [eval_ner(y_true[idx], y_pred[idx]) for idx in range(len(y_true))]
avg_metrics_fold = {key: sum(d[key] for d in scores) / len(scores) for key in scores[0]}
rich.print(avg_metrics_fold)

# Evaluate quality of extracted tokens
print("--- Token-level evaluation results ---")
tokenlevel_results = eval_ner(
    [(w[0], "") for w in flat_golds],
    [(w[0], "") for w in flat_preds],
)
rich.print(tokenlevel_results)


In [ ]:
# Label-wise analysis
from util import print_avg_metrics


y_true_per = [[ent for ent in item if ent[1] == "PER"] for item in y_true]
y_true_loc = [[ent for ent in item if ent[1] == "LOC"] for item in y_true]
y_true_org = [[ent for ent in item if ent[1] == "ORG"] for item in y_true]
y_true_misc = [[ent for ent in item if ent[1] == "MISC"] for item in y_true]

y_pred_per = [[ent for ent in item if ent[1] == "PER"] for item in y_pred]
y_pred_loc = [[ent for ent in item if ent[1] == "LOC"] for item in y_pred]
y_pred_org = [[ent for ent in item if ent[1] == "ORG"] for item in y_pred]
y_pred_misc = [[ent for ent in item if ent[1] == "MISC"] for item in y_pred]

print("--- PER-entities evaluation results ---")
print_avg_metrics(y_true_per, y_pred_per)

print("--- ORG-entities evaluation results ---")
print_avg_metrics(y_true_org, y_pred_org)

print("--- LOC-entities evaluation results ---")
print_avg_metrics(y_true_loc, y_pred_loc)

print("--- MISC-entities evaluation results ---")
print_avg_metrics(y_true_misc, y_pred_misc)

In [ ]:
import seaborn as sns
from util import compute_ner_conf_matrix

conf_matrix = compute_ner_conf_matrix(y_pred, y_true)

labels = ["PER", "LOC", "ORG", "MISC"]
sns.heatmap(
    conf_matrix,
    annot=True,
    cmap="Blues",
    fmt="d",
    xticklabels=labels,
    yticklabels=labels,
)

In [ ]:
tmp = [
    (y_true[idx], y_pred[idx], eval_ner(y_true[idx], y_pred[idx])["f1"])
    for idx in range(len(y_true))
]
tmp = sorted(tmp, key=lambda x: x[2])
for i in tmp:
    print(f"true: {i[0]}")
    print(f"pred: {i[1]}")
    print(f"f1: {i[2]}")
    print()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

f1_scores = [item["f1"] for item in scores]
sns.kdeplot(f1_scores, fill=True, bw_adjust=0.5)
plt.xlabel("F1 Score")
plt.ylabel("Density")
plt.title("F1 Score Distribution (KDE)")
plt.xlim(0, 1)
plt.grid(alpha=0.3)
plt.show()